# Análisis de ingresos y tarifas de telecomunicaciones

**Proyecto académico de Sprint 5 · TripleTen.** Este notebook presenta código utilizado en un ejercicio formativo; no corresponde a un sistema desplegado en producción.

## Objetivo
Consolidación de consumos por usuario y mes, cálculo de ingresos por plan, comparación Surf vs Ultimate y análisis de diferencias regionales con pruebas estadísticas.

## Datos y reproducibilidad
Los archivos originales del ejercicio **no están incluidos** en el repositorio. Para ejecutar el análisis, coloca copias obtenidas legítimamente en una carpeta `data/` junto al notebook. Archivos esperados: megaline_users.csv, megaline_calls.csv, megaline_messages.csv, megaline_internet.csv y megaline_plans.csv.

Los resultados documentados se extrajeron de la ejecución revisada del notebook original; las salidas de celdas se omitieron de esta edición para preservar un documento de código más ligero. El resultado puede variar si se modifican datos, dependencias o versiones de librerías.


In [ ]:
# Cargar todas las librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Carga los archivos de datos en diferentes DataFrames

df_calls = pd.read_csv('data/megaline_calls.csv')
df_internet = pd.read_csv('data/megaline_internet.csv')
df_messages = pd.read_csv('data/megaline_messages.csv')
df_plans = pd.read_csv('data/megaline_plans.csv')
df_users = pd.read_csv('data/megaline_users.csv')

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de las tarifas
print(df_plans.info())

In [ ]:
# Imprime una muestra de los datos para las tarifas
print(df_plans.head())

display(df_plans)

In [ ]:
# No requirió correcciones adicionales.

In [ ]:
# agregar nueva columna gb_per_month_included
df_plans['gb_per_month_included'] = df_plans['mb_per_month_included']/1024
print(df_plans['gb_per_month_included'])

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de usuarios

print(df_users.info())

In [ ]:
# Imprime una muestra de datos para usuarios

print(df_users.head())
display(df_users)

In [ ]:
#cambiar tipo de datos columna reg_date de object a datetime
df_users['reg_date'] = pd.to_datetime(df_users['reg_date'])

# comprobar cuantos datos na tenemos
print(f'Celdas sin datos:', df_users['churn_date'].isna().sum())

#cambio de tipo de variable de object a datetime en churn_date
df_users['churn_date'] = pd.to_datetime(df_users['churn_date'])


#Verificar y comprobar cambios
df_users.info()
display(df_users)

In [ ]:
# creacion de columna en df_users llamada status, con variables active y churned
df_users['status'] = df_users['churn_date'].apply(
    lambda x: 'active' if pd.isna(x) else 'churned')

# comprobacion de creacion de columna y de variables
print(f'Celdas sin datos:', df_users['status'].isna().sum())
print(df_users['status'].unique())

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de las llamadas

print(df_calls.info())

In [ ]:
# Imprime una muestra de datos para las llamadas

print(df_calls.head())

In [ ]:
# convertir variable call_date de object a datetime
df_calls['call_date'] = pd.to_datetime(df_calls['call_date'])
print(df_calls['call_date'].dtype)

In [ ]:
#creacion de columna del mes
df_calls['month'] = df_calls['call_date'].dt.month

#Se eliminaron registros con duración igual a 0, ya que representan llamadas no contestadas que no generan consumo facturable
df_calls = df_calls[df_calls['duration'] > 0]
df_calls['duration'] = np.ceil(df_calls['duration']).astype(int)
display(df_calls)

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de los mensajes

print(df_messages.info())

In [ ]:
# Imprime una muestra de datos para los mensajes

print(df_messages.head())

In [ ]:
# conversion de la variable message_date de object a datetime
df_messages['message_date'] = pd.to_datetime(df_messages['message_date'])
print(df_messages['message_date'].dtype)

In [ ]:
df_messages['month'] = df_messages['message_date'].dt.month

display(df_messages['month'])

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de internet

print(df_internet.info())

In [ ]:
# Imprime una muestra de datos para el tráfico de internet

print(df_internet.head())

In [ ]:
# conversion de tipo de variable de object a datetime en session_date
df_internet['session_date'] = pd.to_datetime(df_internet['session_date'])
print(df_internet['session_date'].dtype)

In [ ]:
df_internet['month']=df_internet['session_date'].dt.month
display(df_internet['month'])

In [ ]:
# Imprime las condiciones de la tarifa y asegúrate de que te quedan claras

display(df_plans)

In [ ]:
# Calcula el número de llamadas hechas por cada usuario al mes. Guarda el resultado.

calls_per_month = df_calls.groupby(['user_id', 'month'])['id'].count().reset_index(name='total_calls')

display(calls_per_month)

In [ ]:
# Calcula la cantidad de minutos usados por cada usuario al mes. Guarda el resultado.

minutes_per_month = df_calls.groupby(['user_id','month'])['duration'].sum().reset_index(name='total_minutes')
display(minutes_per_month)

In [ ]:
# Calcula el número de mensajes enviados por cada usuario al mes. Guarda el resultado.

messages_per_month = df_messages.groupby(['user_id','month'])['id'].count().reset_index(name='total_messages')
display(messages_per_month)

In [ ]:
# Calcula el volumen del tráfico de Internet usado por cada usuario al mes. Guarda el resultado.

mb_per_month = df_internet.groupby(['user_id', 'month'])['mb_used'].sum().reset_index(name='total_mb')
mb_per_month['total_gb'] = np.ceil(mb_per_month['total_mb'] / 1024).astype(int)
display(mb_per_month)

In [ ]:
# Fusiona los datos de llamadas, minutos, mensajes e Internet con base en user_id y month

df_merged = calls_per_month.merge(
    minutes_per_month,
    on=['user_id', 'month'],
    how='outer'
)

df_merged = df_merged.merge(
    messages_per_month,
    on=['user_id', 'month'],
    how='outer'
)

df_merged = df_merged.merge(
    mb_per_month,
    on=['user_id', 'month'],
    how='outer'
)

df_merged = df_merged.fillna(0)

df_merged.info()
display(df_merged)

In [ ]:
# Añade la información de la tarifa

monthly_usage = df_merged.merge(
    df_users[['user_id', 'plan']],
    on='user_id',
    how='left'
)

monthly_usage = monthly_usage.merge(
    df_plans,
    left_on='plan',
    right_on='plan_name',
    how='left'
)

monthly_usage.drop(columns='plan_name', inplace=True)

monthly_usage = monthly_usage.merge(
    df_users[['user_id', 'city']],
    on='user_id',
    how='left'
)
                                   
display(monthly_usage)

In [ ]:
# Calcula el ingreso mensual para cada usuario

# Extra
monthly_usage['extra_minutes'] = (
    monthly_usage['total_minutes']
    - monthly_usage['minutes_included']
).clip(lower=0)

monthly_usage['extra_messages'] = (
    monthly_usage['total_messages']
    - monthly_usage['messages_included']
).clip(lower=0)

monthly_usage['extra_gb'] = (
    monthly_usage['total_gb']
    - monthly_usage['gb_per_month_included']
).clip(lower=0)

#Cargos
monthly_usage['minutes_charge'] = (
    monthly_usage['extra_minutes']
    * monthly_usage['usd_per_minute']
)

monthly_usage['messages_charge'] = (
    monthly_usage['extra_messages']
    * monthly_usage['usd_per_message']
)

monthly_usage['gb_charge'] = (
    monthly_usage['extra_gb']
    * monthly_usage['usd_per_gb']
)

monthly_usage['monthly_revenue'] = (
    monthly_usage['usd_monthly_pay']
    + monthly_usage['minutes_charge']
    + monthly_usage['messages_charge']
    + monthly_usage['gb_charge']
)
monthly_usage.info()
display(monthly_usage)

In [ ]:
# Compara la duración promedio de llamadas por cada plan y por cada mes. Traza un gráfico de barras para visualizarla.

avg_minutes = (
    monthly_usage
    .groupby(['month', 'plan'])['total_minutes']
    .mean()
    .reset_index()
)

avg_minutes_pivot = avg_minutes.pivot(
    index='month',
    columns='plan',
    values='total_minutes'
)

avg_minutes_pivot.plot(kind='bar', figsize=(10,5))

plt.title('Duración promedio de llamadas por mes y plan')
plt.xlabel('Mes')
plt.ylabel('Minutos promedio')
plt.legend(title='Plan')
plt.show()

In [ ]:
# Compara el número de minutos mensuales que necesitan los usuarios de cada plan. Traza un histograma.


surf = monthly_usage[monthly_usage['plan'] == 'surf']['total_minutes']
ultimate = monthly_usage[monthly_usage['plan'] == 'ultimate']['total_minutes']

# Histograma
plt.figure(figsize=(10,5))

plt.hist(surf, bins=20, alpha=0.5)
plt.hist(ultimate, bins=20, alpha=0.5)

plt.title('Distribución de minutos mensuales por plan')
plt.xlabel('Minutos')
plt.ylabel('Frecuencia')
plt.legend(['surf', 'ultimate'])

plt.show()

In [ ]:
# Calcula la media y la varianza de la duración mensual de llamadas.

stats_calls = monthly_usage.groupby('plan')['total_minutes'].agg(['mean','var'])
print(stats_calls)

In [ ]:
# Traza un diagrama de caja para visualizar la distribución de la duración mensual de llamadas

monthly_usage.boxplot(column='total_minutes', by='plan', figsize=(8,5))

plt.title('Distribución de minutos mensuales por plan')
plt.suptitle('')  # elimina el título automático de pandas
plt.xlabel('Plan')
plt.ylabel('Minutos mensuales')
plt.show()

In [ ]:
# Compara el número de mensajes que tienden a enviar cada mes los usuarios de cada plan
avg_messages = (
    monthly_usage
    .groupby(['month', 'plan'])['total_messages']
    .mean()
    .reset_index()
)
avg_messages_pivot = avg_messages.pivot(
    index='month',
    columns='plan',
    values='total_messages'
)

avg_messages_pivot.plot(kind='bar', figsize=(10,5))

plt.title('Promedio de mensajes por mes y plan')
plt.xlabel('Mes')
plt.ylabel('Número promedio de mensajes')
plt.legend(title='Plan')
plt.show()

In [ ]:
# Compara la cantidad de tráfico de Internet consumido por usuarios por plan

avg_internet = (
    monthly_usage
    .groupby(['month', 'plan'])['total_mb']
    .mean()
    .reset_index()
)

avg_internet_pivot = avg_internet.pivot(
    index='month',
    columns='plan',
    values='total_mb'
)

avg_internet_pivot.plot(kind='bar', figsize=(10,5))

plt.title('Consumo promedio de Internet por mes y plan')
plt.xlabel('Mes')
plt.ylabel('Megabytes consumidos')
plt.legend(title='Plan')
plt.show()

In [ ]:
#mensajes por plan
monthly_usage.boxplot(column='total_messages', by='plan', figsize=(8,5))
plt.title('Mensajes por plan')
plt.suptitle('')
plt.show()

In [ ]:
print(monthly_usage.groupby('plan')['total_mb'].mean())

In [ ]:
monthly_usage[monthly_usage['plan']=='surf']['total_mb'].hist(alpha=0.5, bins=20)
monthly_usage[monthly_usage['plan']=='ultimate']['total_mb'].hist(alpha=0.5, bins=20)

plt.legend(['surf','ultimate'])
plt.title('Consumo de internet por plan')
plt.show()

In [ ]:
monthly_usage.boxplot(column='total_mb', by='plan', figsize=(8,5))
plt.title('Consumo de Internet por plan')
plt.suptitle('')
plt.show()

In [ ]:
revenue_stats = monthly_usage.groupby('plan')['monthly_revenue'].agg(['mean','var','std'])
print(revenue_stats)

In [ ]:
monthly_usage.boxplot(column='monthly_revenue', by='plan', figsize=(8,5))

plt.title('Distribución de ingresos por plan')
plt.suptitle('')
plt.xlabel('Plan')
plt.ylabel('Ingresos mensuales')
plt.show()

In [ ]:
monthly_usage[monthly_usage['plan']=='surf']['monthly_revenue'].hist(alpha=0.5, bins=20)
monthly_usage[monthly_usage['plan']=='ultimate']['monthly_revenue'].hist(alpha=0.5, bins=20)

plt.legend(['surf','ultimate'])
plt.title('Distribución de ingresos mensuales por plan')
plt.show()

In [ ]:
# Prueba las hipótesis
surf_revenue = monthly_usage[monthly_usage['plan'] == 'surf']['monthly_revenue']
ultimate_revenue = monthly_usage[monthly_usage['plan'] == 'ultimate']['monthly_revenue']

alpha=0.05

result = stats.ttest_ind(surf_revenue, ultimate_revenue, equal_var=False)

if result.pvalue < alpha:
    print("Rechazamos H0: sí hay diferencia significativa")
else:
    print("No rechazamos H0: no hay evidencia de diferencia")

In [ ]:
# Prueba las hipótesis

monthly_usage['region'] = np.where(
    monthly_usage['city'].str.contains('NY-NJ|New York|Newark|Jersey'),
    'NY_NJ',
    'other'
)

alpha = 0.05

ny_nj = monthly_usage[monthly_usage['region'] == 'NY_NJ']['monthly_revenue']
other = monthly_usage[monthly_usage['region'] == 'other']['monthly_revenue']

result = stats.ttest_ind(ny_nj, other, equal_var=False)

if result.pvalue < alpha:
    print("Rechazamos H0: hay diferencia significativa en ingresos")
else:
    print("No rechazamos H0: no hay evidencia de diferencia")

## Resultado documentado

En los datos analizados, las pruebas t de Welch permitieron rechazar la hipótesis de igualdad de ingresos promedio entre planes y entre NY-NJ y el resto de regiones al nivel de significancia del 5%.

## Alcance
Este notebook es evidencia de habilidades aplicadas en un ejercicio académico de TripleTen. No incluye datasets originales ni registros de usuarios reales del IMSS.
